In [1]:
import subprocess
import re
import numpy as np
import itertools
import json


def build_command(l1, l2, l3, l4):
    return [
        "python",
        "run_exam_scheduler.py", "--no-repair",
        "--input-csv", "../Student Course (Jul-Nov 2025 and Winter 2025) Makeup Theory.csv",
        "--adjacency-mode", "major",
        "--k", "14",
        "--num-reads", "10",
        "--lambda1", str(l1),
        "--lambda2", str(l2),
        "--lambda3", str(l3),
        "--lambda4", str(l4),
        "--capacity", "5500"
    ]


def parse_metrics(stdout_text):
    patterns = {
        "c1": r"C1\s+one-hot violations:\s*(\d+)",
        "c2": r"C2\s+same-slot conflict violations:\s*(\d+)",
        "c3": r"C3\s+consecutive-slot violations:\s*(\d+)",
        "c4_over": r"C4\s+slots over capacity:\s*(\d+)",
        "c4_total_overflow": r"C4\s+total overflow:\s*([-\d\.]+)",
        "assigned": r"Exams assigned:\s*(\d+)\s*/\s*(\d+)",
        "energy": r"Energy:\s*([-\d\.]+)",
        "valid": r"Valid:\s*(.+)"
    }

    result = {}

    for key, pat in patterns.items():
        m = re.search(pat, stdout_text)
        if not m:
            result[key] = None
        else:
            if key == "assigned":
                result["assigned_now"] = int(m.group(1))
                result["assigned_total"] = int(m.group(2))
            elif key in ["c4_total_overflow", "energy"]:
                result[key] = float(m.group(1))
            elif key == "valid":
                result[key] = m.group(1).strip()
            else:
                result[key] = int(m.group(1))

    return result


def compute_loss(metrics):
    c1 = metrics["c1"] if metrics["c1"] is not None else 10**6
    c2 = metrics["c2"] if metrics["c2"] is not None else 10**6
    c3 = metrics["c3"] if metrics["c3"] is not None else 10**6
    c4 = metrics["c4_over"] if metrics["c4_over"] is not None else 10**6

    assigned_now = metrics.get("assigned_now", 0) or 0
    assigned_total = metrics.get("assigned_total", 722) or 722
    unassigned = assigned_total - assigned_now

    return c1 + c2 + c3 + c4 


def run_solver(lambdas):
    l1, l2, l3, l4 = lambdas
    cmd = build_command(l1, l2, l3, l4)

    proc = subprocess.run(cmd, capture_output=True, text=True)
    stdout_text = proc.stdout + "\n" + proc.stderr

    metrics = parse_metrics(stdout_text)
    loss = compute_loss(metrics)

    return {
        "lambdas": [float(l1), float(l2), float(l3), float(l4)],
        "loss": float(loss),
        "metrics": metrics,
        "raw_output": stdout_text,
        "returncode": int(proc.returncode),
    }


def run_solver_log(theta):
    lambdas = np.exp(theta)
    return run_solver(lambdas)


def average_run_solver_log(theta, repeats=1):
    results = []
    for _ in range(repeats):
        results.append(run_solver_log(theta))

    avg_loss = float(np.mean([r["loss"] for r in results]))
    best_rep = min(results, key=lambda x: x["loss"])

    return {
        "loss": avg_loss,
        "metrics": best_rep["metrics"],
        "lambdas": best_rep["lambdas"],
        "raw_output": best_rep["raw_output"],
        "returncode": int(best_rep["returncode"]),
    }


def full_sign_search(theta, ck, repeats=1, theta_min=-20, theta_max=20):
    best = None
    best_delta = None

    for delta_tuple in itertools.product([-1.0, 1.0], repeat=4):
        delta = np.array(delta_tuple, dtype=float)
        candidate = np.clip(theta + ck * delta, theta_min, theta_max)

        res = average_run_solver_log(candidate, repeats=repeats)

        if best is None or res["loss"] < best["loss"]:
            best = dict(res)
            best_delta = delta.copy()
            best["theta"] = candidate.tolist()

    return best, best_delta.tolist()


def to_jsonable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    return obj


def hybrid_spsa_fullsign(
    init_lambdas,
    num_iters=20,
    a=0.18,
    c=0.5,
    A=10.0,
    alpha=0.602,
    gamma=0.101,
    grad_clip=2.0,
    theta_min=-20,
    theta_max=20,
    spsa_repeats=1,
    full_repeats=1,
    full_search_every=5,
    seed=42,
    save_json_path="hybrid_lambda_history.json"
):
    rng = np.random.default_rng(seed)
    theta = np.log(np.array(init_lambdas, dtype=float))

    current = average_run_solver_log(theta, repeats=spsa_repeats)
    best_result = dict(current)
    best_theta = theta.copy()

    history = []

    for k in range(num_iters):
        ak = a / ((k + 1 + A) ** alpha)
        ck = c / ((k + 1) ** gamma)

        used_full_search = ((k + 1) % full_search_every == 0)

        if used_full_search:
            full_best, best_delta = full_sign_search(
                theta, ck,
                repeats=full_repeats,
                theta_min=theta_min,
                theta_max=theta_max
            )

            theta_candidate = np.array(full_best["theta"], dtype=float)
            current_result = dict(full_best)

            move_type = "FULL_SIGN_SEARCH"
            extra_info = {
                "best_delta": best_delta
            }

        else:
            delta = rng.choice([-1.0, 1.0], size=theta.shape)

            theta_plus = np.clip(theta + ck * delta, theta_min, theta_max)
            theta_minus = np.clip(theta - ck * delta, theta_min, theta_max)

            res_plus = average_run_solver_log(theta_plus, repeats=spsa_repeats)
            res_minus = average_run_solver_log(theta_minus, repeats=spsa_repeats)

            y_plus = float(res_plus["loss"])
            y_minus = float(res_minus["loss"])

            ghat = (y_plus - y_minus) / (2.0 * ck * delta)
            ghat = np.clip(ghat, -grad_clip, grad_clip)

            theta_candidate = np.clip(theta - ak * ghat, theta_min, theta_max)
            current_result = average_run_solver_log(theta_candidate, repeats=spsa_repeats)

            move_type = "SPSA"
            extra_info = {
                "delta": delta.tolist(),
                "y_plus": y_plus,
                "y_minus": y_minus,
                "ghat": ghat.tolist()
            }

        theta = theta_candidate

        if current_result["loss"] < best_result["loss"]:
            best_result = dict(current_result)
            best_theta = theta.copy()

        log_row = {
            "iter": int(k + 1),
            "move_type": move_type,
            "ak": float(ak),
            "ck": float(ck),
            "theta": theta.tolist(),
            "lambdas": np.exp(theta).tolist(),
            "loss": float(current_result["loss"]),
            "metrics": current_result["metrics"],
            "best_loss_so_far": float(best_result["loss"]),
            **extra_info
        }
        history.append(log_row)

        print("=" * 72)
        print(f"Iteration {k+1}/{num_iters}   [{move_type}]")
        print(f"ak: {ak:.6f}  ck: {ck:.6f}")
        print("Lambdas:", np.exp(theta))
        print("Loss:", current_result["loss"])
        print("Metrics:", current_result["metrics"])
        print("Best loss so far:", best_result["loss"])
        '''
        with open(save_json_path, "w", encoding="utf-8") as f:
            json.dump(
                to_jsonable({
                    "best_lambdas": np.exp(best_theta).tolist(),
                    "best_result": best_result,
                    "history": history
                }),
                f,
                indent=2
            )
       '''
    return best_result, np.exp(best_theta), history

In [2]:
def adaptive_init_relative(initial_lambdas):
    result = run_solver(initial_lambdas)
    metrics = result["metrics"]

    c = np.array([
        metrics["c1"] or 1,
        metrics["c2"] or 1,
        metrics["c3"] or 1,
        metrics["c4_over"] or 1
    ], dtype=float)

    # normalize violations
    c_norm = c / (np.max(c) + 1e-6)

    base = 10000

    new_lambdas = base * c_norm + 1e-6

    print("\n🔹 Relative scaled lambdas:", new_lambdas)

    return new_lambdas.tolist()

In [3]:
init_lambdas = [10000, 8000, 4000, 0.01]

# adaptive scaling
#scaled_lambdas = adaptive_init_relative(init_lambdas)



In [4]:
best_result, best_lambdas, history = hybrid_spsa_fullsign(
    init_lambdas=init_lambdas,
    num_iters=30,
    a=0.18,
    c=0.5,
    grad_clip=2.0,
    full_search_every=5,   # every 5th iteration do all 16 sign combinations
    spsa_repeats=1,
    full_repeats=1,
    seed=123
)

print("\nBEST LAMBDAS:", best_lambdas)
print("BEST LOSS:", best_result["loss"])
print("BEST METRICS:", best_result["metrics"])

Iteration 1/30   [SPSA]
ak: 0.042497  ck: 0.500000
Lambdas: [1.e+04 8.e+03 4.e+03 1.e-02]
Loss: 4000000.0
Metrics: {'c1': None, 'c2': None, 'c3': None, 'c4_over': None, 'c4_total_overflow': None, 'assigned': None, 'energy': None, 'valid': None}
Best loss so far: 4000000.0
Iteration 2/30   [SPSA]
ak: 0.040328  ck: 0.466193
Lambdas: [1.e+04 8.e+03 4.e+03 1.e-02]
Loss: 4000000.0
Metrics: {'c1': None, 'c2': None, 'c3': None, 'c4_over': None, 'c4_total_overflow': None, 'assigned': None, 'energy': None, 'valid': None}
Best loss so far: 4000000.0
Iteration 3/30   [SPSA]
ak: 0.038431  ck: 0.447487
Lambdas: [1.e+04 8.e+03 4.e+03 1.e-02]
Loss: 4000000.0
Metrics: {'c1': None, 'c2': None, 'c3': None, 'c4_over': None, 'c4_total_overflow': None, 'assigned': None, 'energy': None, 'valid': None}
Best loss so far: 4000000.0
Iteration 4/30   [SPSA]
ak: 0.036754  ck: 0.434672
Lambdas: [1.e+04 8.e+03 4.e+03 1.e-02]
Loss: 4000000.0
Metrics: {'c1': None, 'c2': None, 'c3': None, 'c4_over': None, 'c4_total_ov

KeyboardInterrupt: 

In [5]:
!python run_exam_scheduler.py   --input-csv "../Student Course (Jul-Nov 2025 and Winter 2025) Makeup Theory.csv"   --adjacency-mode major   --k 14 --num-reads 1  --lambda1 5.82901813e+03 --lambda2 4.82041924e+03 --lambda3 1.20661709e+04 --lambda4 9.50083477e-03 --capacity 5500 --visualize

⚠ D-Wave Ocean SDK not available. Neal backend disabled.

EXAM SCHEDULING PIPELINE
Input CSV: ../Student Course (Jul-Nov 2025 and Winter 2025) Makeup Theory.csv
Adjacency mode: major
K (time slots): 14
Backend: NEAL
λ1=5829.01813  λ2=4820.41924  λ3=12066.1709  λ4=0.00950083477
Slot capacity: 5500

GENERATING DATASET FROM CSV
Input CSV: ../Student Course (Jul-Nov 2025 and Winter 2025) Makeup Theory.csv
Adjacency mode for solver: major
✓ Rows after filters (all): 8655
✓ Rows after filters (major): 8655
✓ Major graph: 717 courses, 3382 edges (1.32% density)
✓ All graph: 717 courses, 3382 edges (1.32% density)
✓ Selected 'major' graph for solver
✓ Saved dataset to: output/run_20260330_063957/major

RUNNING SOLVER FOR MODE: MAJOR

GENERATING VISUALIZATIONS

✓ Saved adjacency heatmap to: output/run_20260330_063957/major/adjacency_heatmap.png
✓ Saved conflict graph to: output/run_20260330_063957/major/conflict_graph.png

BUILDING QUBO MATRIX
Exams: 717
Colors (K): 14
Exam variables: 10038
Sla

In [36]:
!df -h

Filesystem                                                           Size  Used Avail Use% Mounted on
overlay                                                              492G  200G  267G  43% /
tmpfs                                                                 64M     0   64M   0% /dev
/dev/mapper/ubuntu--vg-lv--0                                         492G  200G  267G  43% /etc/hosts
shm                                                                   64M  4.0K   64M   1% /dev/shm
172.17.111.25:/trident_nas_pvc_e1326982_f088_46bc_a19a_1c8b1d7e3037  251G  169G   82G  68% /home/jovyan/work
tmpfs                                                                1.4T   12K  1.4T   1% /run/secrets/kubernetes.io/serviceaccount
tmpfs                                                                693G   12K  693G   1% /proc/driver/nvidia
/dev/mapper/ubuntu--vg-lv--2                                          90G   14G   72G  16% /usr/bin/nvidia-smi
tmpfs                                      